# Artist/Album Supervision for VarLen Semantic IDs

Короткий notebook для запуска course-project эксперимента.

**Research question:** может ли supervision по артисту и альбому улучшить variable-length semantic IDs для музыкальной рекомендации?

Сравниваем четыре варианта на уменьшенном Yambda subset:

1. `fixed` — fixed-length dVAE baseline.
2. `original` — обычный VarLen dVAE baseline.
3. `aux` — VarLen dVAE + auxiliary artist/album classification loss по полному SID-представлению.
4. `prefix` — VarLen dVAE + prefix-level supervision: первый токен тянем к artist cluster, первые два токена к album cluster.

Notebook не содержит отдельной логики эксперимента: он вызывает скрипты из `scripts/RQ_album_artist_anchor/`, чтобы запуск оставался воспроизводимым из CLI.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run this notebook from the repository root")

def run(cmd, *, check=True):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=ROOT, check=check)

PY = sys.executable
DATA_DIR = ROOT / "data" / "RQ_album_artist_anchor" / "yambda"
RESULTS_DIR = ROOT / "results" / "RQ_album_artist_anchor"
DATA_DIR, RESULTS_DIR

## 1. Download Yambda inputs

Эту ячейку нужно запускать только если `data/yambda/` еще не содержит:

- `interactions.parquet`
- `embeddings.parquet`
- `artist_item_mapping.parquet`
- `album_item_mapping.parquet`

Скачивание требует `datasets` и доступ к Hugging Face.

In [ ]:
# Optional: uncomment for a fresh machine.
# from scripts.data.yambda import download, download_metadata
# download(dst_dir="./data/yambda")
# download_metadata(dst_dir="./data/yambda")

## 2. Build project subset and metadata

По умолчанию берется subset примерно в 4 раза меньше full Yambda setup из статьи: до 200k пользователей, до 20M interactions и до 67k core items. Для совсем маленькой проверки можно уменьшить параметры прямо в команде.

In [ ]:
run([
    PY, "-m", "scripts.RQ_album_artist_anchor.build_yambda_subset",
    "--src-dir", "./data/yambda",
    "--dst-dir", "./data/RQ_album_artist_anchor/yambda",
    "--num-users", "200000",
    "--max-interactions", "20000000",
    "--max-core-items", "67000",
    "--seqrec-test-user-fraction", "0.05",
])

run([
    PY, "-m", "scripts.RQ_album_artist_anchor.build_artist_album_metadata",
    "--src-dir", "./data/yambda",
    "--data-dir", "./data/RQ_album_artist_anchor/yambda",
    "--num-artist-classes", "512",
    "--num-album-classes", "1024",
])

## 3. Tiny smoke run

Если окружение и GPU готовы, сначала прогоняем дешевую baseline-проверку. В текущем lightweight окружении автора smoke test может не запускаться из-за отсутствующих `torch/polars/pyarrow/PyYAML`.

In [ ]:
run([PY, "-m", "scripts.train_dvae", "--config", "configs/RQ_album_artist_anchor/original_varlen_dvae_tiny.yaml"])
run([PY, "-m", "scripts.train_seqrec", "--config", "configs/RQ_album_artist_anchor/seqrec_original_tiny.yaml"])
run([
    PY, "-m", "scripts.RQ_album_artist_anchor.analyze_prefix_purity",
    "--sids", "./results/RQ_album_artist_anchor/original_tiny/sids.parquet",
    "--output", "./results/RQ_album_artist_anchor/original_tiny/prefix_purity.json",
])

## 4. Full experiment runs

Каждый метод пишет:

- `results/RQ_album_artist_anchor/<method>/sids.parquet`
- `results/RQ_album_artist_anchor/<method>/dvae_metrics.json`
- `results/RQ_album_artist_anchor/<method>/seqrec_summary.json`
- `results/RQ_album_artist_anchor/<method>/prefix_purity.json`

Seqrec configs считают `@10`, `@50` и `@100`. Если времени мало, можно запустить `prefix` только до structural diagnostics: `--stages dvae,purity`.

In [ ]:
for method in ["fixed", "original", "aux", "prefix"]:
    run([PY, "-m", "scripts.RQ_album_artist_anchor.run_experiment", "--method", method])

## 5. Collect comparison table

Собираем компактную таблицу с seqrec metrics, purity и collision statistics.

In [ ]:
run([PY, "-m", "scripts.RQ_album_artist_anchor.collect_results"])

comparison_path = RESULTS_DIR / "comparison.json"
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text())
    comparison
else:
    print("comparison.json is not created yet")

## Useful partial reruns

```bash
python3 -m scripts.RQ_album_artist_anchor.run_experiment --method fixed --stages dvae,purity,seqrec
python3 -m scripts.RQ_album_artist_anchor.run_experiment --method original --stages dvae,purity
python3 -m scripts.RQ_album_artist_anchor.run_experiment --method aux --stages seqrec
python3 -m scripts.RQ_album_artist_anchor.run_experiment --method prefix --stages dvae,purity
python3 -m scripts.RQ_album_artist_anchor.collect_results
```

Fallback policy: если полный seqrec слишком долгий, оставляем end-to-end для `original` и `aux`, а для `prefix` репортим structural metrics: artist/album prefix purity, collisions, mean SID length.